<div style="display:flex;gap:14px;align-items:center;flex-wrap:wrap;
 font-family:'Segoe UI',system-ui,sans-serif;font-size:13px;padding:10px 2px;
 border-bottom:2px solid #1B7A43;margin-bottom:4px;">
 <a href="https://colab.research.google.com/github/STG17-Africa/stg17-workshop/blob/main/notebooks/day1/D1_Agent_FR_open.ipynb" target="_blank"><img
  src="https://colab.research.google.com/assets/colab-badge.svg" alt="Ouvrir dans Colab"></a>
 <span style="color:#6B7B75;">Jour 1 · piste ouverte</span>
 <span style="flex:1;"></span>
 <a href="./D1_Agent_EN_open.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">🌐 English</a>
 <a href="./D1_Agent_FR.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">⇄ piste guidée</a>
</div>

<!-- Du RAG à l'agent - Où se place l'humain · STG17 workshop · AfDB / STATAFRIC -->
<!-- GENERATED FILE — edit notebooks/_masters/d1_agent.master.ipynb instead. -->


<div style="background:linear-gradient(135deg,#0B2545 0%,#1B7A43 100%);
 border-radius:18px;padding:32px 38px;font-family:'Segoe UI',system-ui,sans-serif;margin-bottom:6px;">
 <div style="color:#F2A900;font-size:12.5px;letter-spacing:3px;font-weight:700;
  text-transform:uppercase;">Banque africaine de développement · UA STATAFRIC · STG17 · Jour 1 · 15h45</div>
 <div style="color:#fff;font-size:2em;font-weight:800;margin:10px 0 8px;line-height:1.15;">
  Du RAG à l'agent</div>
 <div style="color:#dbe7e0;font-size:1.05em;line-height:1.55;max-width:900px;">
  Ce matin, le modèle recevait des passages que vous aviez choisis. Il choisit maintenant quel
  outil appeler. Une chose ne change pas : <b>le modèle demande, votre code exécute</b> — et tout
  ce que votre office doit contrôler vit dans cet interstice.
 </div>
 <div style="color:#F2A900;font-size:13px;margin-top:14px;font-weight:600;">
  75 minutes · les étapes centrales fonctionnent sans aucune clé API</div>
</div>

> **Pourquoi un agent est un risque différent.** Un assistant qui se trompe produit
> une phrase fausse, et un humain la lit avant qu'elle n'aille où que ce soit. Un
> agent qui se trompe accomplit une **action** fausse. L'échec a déjà eu lieu quand
> quelqu'un lit la sortie.


### Le parcours de ce laboratoire

| # | Étape | Ce que vous apprenez |
|---|-------|----------------------|
| 1 | La boîte à outils | Ce qu'est un outil, et pourquoi `writes` est un champ et non un commentaire |
| 2 | Le protocole | Comment un modèle demande une chose qu'il ne peut pas exécuter |
| 3 | **L'interstice** | Les vingt lignes où vit la politique de votre office |
| 4 | La boucle, sans modèle | Tous les modes d'échec, reproductibles, sans clé API |
| 5 | La boucle, pour de vrai | Le même code, piloté par un vrai modèle |
| 6 | L'outil d'écriture | Le refuser, puis l'approuver, et observer le disque |
| 7 | Le budget | Pourquoi `max_steps` est un contrôle de coût, pas un filet de sécurité |
| 8 | Le journal d'audit | Ce que vous montrez quand on demande ce qui s'est passé |

**Prérequis.** La boîte à outils `stg17` et `scikit-learn`. **Les étapes 1 à 4 et 6
à 8 n'exigent aucun fournisseur de modèle** — elles exercent la part que votre
office écrit et possède. L'étape 5 en exige un.

> C'est délibéré. La boucle, le contrôle, la gestion d'erreur et le journal sont à
> vous. Le modèle est loué. Construire la vôtre d'abord, et la tester sans la part
> louée, est tout l'enjeu méthodologique de cette session.


In [ ]:
# La boîte à outils, et l'index de récupération construit ce matin.
import subprocess
import sys

REPO = "https://github.com/STG17-Africa/stg17-workshop"

try:
    import stg17  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO}.git"], check=False)

import pandas as pd  # noqa: E402
from stg17 import agent, countries, env, llm, rag, ui  # noqa: E402
from stg17.i18n import T  # noqa: E402

S = env.setup({"scikit-learn": "scikit-learn>=1.3", "pandas": "pandas>=2.0"}, lang="FR")

COUNTRY_ISO3 = "CIV"
C = countries.get(COUNTRY_ISO3)
OUT = S.outputs(C.iso3, "d1_agent")

CORPUS_DIR = rag.write_sample_corpus(S.path("corpus", "sample"))
docs = rag.load_corpus(CORPUS_DIR)
index = rag.build_index(rag.chunk_documents(docs), kind="tfidf")
print(f"{C.name_en} ({C.iso3}) · {len(docs)} documents · -> {OUT}")

---
## 1 · La boîte à outils

Un outil est une fonction que vous autorisez le modèle à demander. Quatre ici :
trois qui lisent, un qui écrit.

`writes` est un champ de l'outil, pas une note dans la documentation. Un outil qui
ne fait que lire peut s'exécuter sans demander à personne. Un outil qui modifie
quelque chose — un fichier, un enregistrement, un courriel, un paiement — est un
risque d'une autre nature. En faire une propriété typée fait qu'un office ne peut
pas oublier lequel est lequel en ajoutant le cinquième outil dans six mois.


In [ ]:
# Trois outils de lecture et un d'écriture. Lisez les signatures que verra le modèle.
TOOLS = [
    agent.search_tool(index),
    agent.list_tool(docs),
    agent.calculator_tool(),
    agent.note_tool(OUT / "notes"),
]

print(agent.render_tools(TOOLS))

ui.result_table(
    pd.DataFrame([{"tool": t.name, "writes": t.writes,
                   "default policy": "refused" if t.writes else "runs"} for t in TOOLS]),
    caption=T("The default policy runs reads and refuses writes. You will change it in step 6.",
              "La politique par défaut exécute les lectures et refuse les écritures. "
              "Vous la changerez à l'étape 6."),
)

### Pourquoi une calculatrice, si le modèle sait calculer ?

Il ne sait pas. Un modèle de langage prédit le token suivant : il prédit donc des
*chiffres* qui ont l'air justes plutôt que de calculer — et il se trompe avec
assurance sur des nombres qui comptent pour un office statistique.

Notez le garde-fou dans cet outil : il accepte les chiffres et les opérateurs et
refuse tout le reste. C'est une liste blanche, pas une liste noire. Exécuter
`eval` sur la sortie d'un modèle avec moins que cela est un chemin d'exécution de
code à distance, et « le modèle ne ferait pas ça » n'est pas un contrôle de
sécurité.


In [ ]:
# Le garde-fou, démontré. Les deux sont refusés, pas exécutés.
calc = agent.calculator_tool()
for expression in ["19.2 - 8.6", "__import__('os').system('echo hello')"]:
    print(f"{expression[:44]:<46} -> {calc.run(expression=expression)[:60]}")

---
## 2 · Le protocole — comment un modèle demande

Le modèle n'a aucune capacité d'exécution. Il émet du texte. Nous convenons d'une
forme pour ce texte, et notre code la lit.

Lisez le prompt système ci-dessous, puis la dernière règle en particulier :
*n'énoncez jamais un chiffre qu'un outil ne vous a pas renvoyé*. C'est la règle de
refus de ce matin, déplacée dans un cadre où la conséquence est plus grande.


In [ ]:
# Ce qui est dit au modèle. La liste d'outils est générée depuis TOOLS.
print(agent.SYSTEM.replace("<<TOOLS>>", agent.render_tools(TOOLS))[:1500])

In [ ]:
# L'analyse tolère ce que font réellement les modèles, et reste stricte sur le reste.
EXAMPLES = [
    '{"thought":"look it up","tool":"search_documents","args":{"query":"unemployment"}}',
    '```json\n{"tool":"calculate","args":{"expression":"2+2"}}\n```',
    'Certainly! {"answer":"8.6 per cent [search_documents]"} Let me know if...',
    'I will now search the documents for you.',
]
for text in EXAMPLES:
    try:
        action = agent.parse_action(text)
        what = f"answer: {action.answer[:34]}" if action.is_final else f"tool: {action.tool}"
        print(f"  OK      {text[:44]!r:<48} -> {what}")
    except agent.ProtocolError as exc:
        print(f"  REJECT  {text[:44]!r:<48} -> {exc}")

> **Le dernier exemple n'est pas un échec du modèle.** C'est une réponse sur
> laquelle notre code ne peut pas agir, et l'agent renvoie l'erreur au modèle pour
> qu'il se corrige. Un système qui plante sur une réponse malformée plantera en
> production, car les modèles en produisent.


---
## 3 · L'interstice

Le voici en entier, extrait de `stg17/agent.py` :

```python
# Le modèle a demandé. Rien ne s'est encore produit.
allowed = self.approve(tool, step.action.args)
step.approved = allowed
if not allowed:
    step.error = "refused by policy"
    transcript.append("TOOL RESULT: REFUSED — un humain n'a pas approuvé cet appel.")
    continue

result = tool.run(**step.action.args)
```

Quatre lignes entre la demande et l'exécution. Tout point d'approbation,
obligation d'audit et contrôle de sûreté de votre office s'y place. Il n'y a nulle
part ailleurs où le mettre, et cela ne s'ajoute pas après coup sans modifier cette
boucle.

Trois politiques sont fournies :

| Politique | Comportement |
|---|---|
| `approve_reads_only` | Les lectures s'exécutent. Les écritures sont refusées. **Par défaut.** |
| `approve_interactive` | Les lectures s'exécutent. Une écriture demande à une personne. |
| `approve_all` | Tout s'exécute. Correct sur un corpus d'exemple ; faux sur un enregistrement. |


---
## 4 · La boucle, pilotée par un script

`ScriptedModel` rejoue des réponses fixes et ne décide rien. C'est précisément ce
qui le rend utile : il permet de reproduire chaque mode d'échec à volonté, et
n'exige aucune clé API.

Le script ci-dessous contient délibérément un exemplaire de chaque chose qui
tourne mal : un bon appel, une réponse encadrée, un outil inexistant, une réponse
illisible, une écriture refusée, et enfin une réponse.


In [ ]:
# À FAIRE: Exécutez l'agent avec le modèle scripté et affichez son journal d'audit
...

In [ ]:
# L'écriture refusée n'a-t-elle vraiment pas eu lieu ? Vérifiez le disque, pas le journal.
notes = OUT / "notes"
print(T(f"note directory exists: {notes.exists()}",
        f"le répertoire de notes existe : {notes.exists()}"))

ui.key_concept(T(
    "A gate that logs a refusal and executes anyway is worse than no gate, because it "
    "produces a reassuring audit trail. Always verify the effect, not the record of it.",
    "Un contrôle qui journalise un refus et exécute quand même est pire que pas de "
    "contrôle, car il produit une trace d'audit rassurante. Vérifiez toujours l'effet, "
    "pas sa trace."))

---
## 5 · La boucle, pilotée par un vrai modèle

Le même `Agent`, les mêmes outils, le même contrôle. Seule la source des réponses
change.

Si aucun fournisseur n'est disponible, passez et poursuivez — vous avez déjà vu le
mécanisme, et les étapes 6 à 8 n'en ont pas besoin.


In [ ]:
# À FAIRE: Exécutez le même agent avec un vrai fournisseur et comparez le journal d'audit
...

---
## 6 · L'outil d'écriture, autorisé

Changez maintenant un seul argument. Rien d'autre ne change dans l'agent — c'est
l'essentiel : la politique est un paramètre, pas une réécriture.

`approve_interactive` vous interrogerait au clavier. Dans un carnet, cela bloque
sur une saisie ; la cellule ci-dessous utilise donc `approve_all` puis vérifie le
disque. Dans votre office, la version interactive est la valeur par défaut honnête
tant que vous n'avez pas de politique écrite disant le contraire.


In [ ]:
# Un seul argument change. Observez le journal et le système de fichiers.
allowed = agent.Agent(
    TOOLS,
    model=agent.ScriptedModel([
        '{"thought":"save the finding","tool":"save_note","args":'
        '{"filename":"finding.md","text":"Youth unemployment exceeds the overall rate '
        'by 10.6 points (Labour Force Survey 2023 Q4, fictional sample corpus)."}}',
        '{"thought":"done","answer":"Saved. [save_note]"}',
    ]),
    approve=agent.approve_all,
).run(T("Save the finding as a note.", "Enregistrez le constat dans une note."))

written = sorted((OUT / "notes").glob("*")) if (OUT / "notes").exists() else []
print(T(f"files now on disk: {[p.name for p in written]}",
        f"fichiers désormais sur disque : {[p.name for p in written]}"))
if written:
    print("-" * 60)
    print(written[0].read_text(encoding="utf-8"))

---
## 7 · Le budget

Un agent décide combien d'appels au modèle il effectue. Sans plafond, le coût
d'une question est illimité — et une boucle incapable de se terminer ne se
terminera pas d'elle-même.

`max_steps` n'est pas un filet de sécurité. C'est un budget, et c'est la seule
raison pour laquelle vous pouvez chiffrer une question avant de la poser.


In [ ]:
# Un modèle qui ne cesse jamais de demander. C'est le plafond qui y met fin.
runaway = agent.Agent(
    TOOLS, max_steps=3,
    model=agent.ScriptedModel(['{"tool":"list_documents","args":{}}'] * 20),
).run(T("Keep going forever.", "Continuez indéfiniment."), verbose=False)

print(T(f"stopped after {len(runaway.steps)} steps · reason: {runaway.stopped}",
        f"arrêté après {len(runaway.steps)} étapes · motif : {runaway.stopped}"))
print(runaway.answer)

---
## 8 · Le journal d'audit — le livrable

Un agent dont les appels d'outils n'ont pas été enregistrés ne peut pas être
expliqué après coup, et un office statistique ne peut pas assumer un processus
qu'il ne peut pas expliquer.

Le journal conserve ce que le modèle a réellement dit, pas un résumé. Quand
quelque chose ira mal dans trois mois, c'est la réponse brute qui dira si le
modèle a demandé la mauvaise chose ou si votre code a mal traité une demande
raisonnable.


In [ ]:
# Tout ce laboratoire, sur disque, prêt pour vendredi.
for name, this_run in [("agent_audit_scripted.json", run),
                       ("agent_audit_approved.json", allowed),
                       ("agent_audit_live.json", live)]:
    if this_run is None:
        continue
    agent.save_audit(this_run, OUT / name, meta={
        "country": C.iso3,
        "tools": [t.name for t in TOOLS],
        "write_tools": [t.name for t in TOOLS if t.writes],
        "policy": "approve_all" if this_run is allowed else "approve_reads_only",
        "max_steps": 8,
        "corpus": str(CORPUS_DIR),
    })
    print(f"  {name}")

run.audit().to_csv(OUT / "agent_audit.csv", index=False)
print(T(f"\nWritten to {OUT}", f"\nÉcrit dans {OUT}"))

---
### Ce que vous avez

<table>
<tr><td><b>agent_audit_*.json</b></td><td>Chaque demande du modèle, son autorisation ou non,
le retour obtenu, et la réponse brute derrière chacune.</td></tr>
<tr><td><b>agent_audit.csv</b></td><td>Le même journal sous forme de tableau, pour un rapport.</td></tr>
<tr><td><b>notes/finding.md</b></td><td>Ce que l'agent a écrit, une fois qu'une politique l'a autorisé.</td></tr>
</table>

### Les limites de ce que vous avez construit

- Le protocole est textuel, pas de l'appel de fonctions natif. Les systèmes de
  production utilisent l'API d'outils du fournisseur — celui-ci est visible à
  dessein, et il demande parfois une reprise quand le modèle enrobe son JSON de prose.
- `approve_all` a servi à l'étape 6 pour la démonstration. C'est la mauvaise valeur
  par défaut pour tout ce qui touche un enregistrement réel.
- L'agent a quatre outils dont un écrit. Ajouter le cinquième est le moment où les
  offices se mettent en difficulté : **chaque nouvel outil est une nouvelle chose
  que le modèle peut demander.**
- **Rien ici ne valide que la réponse découle des résultats d'outils.** Le journal
  prouve ce qui s'est passé, pas que c'était juste.

### Point de contrôle

| Question | Réponse |
|---|---|
| Où se place un point d'approbation ? | Entre la demande du modèle et `tool.run` — nulle part ailleurs |
| Pourquoi `writes` est-il un champ de `Tool` ? | Pour que la politique ne puisse pas oublier quels outils modifient quelque chose |
| Que contrôle `max_steps` ? | Le coût. Un agent décide combien d'appels effectuer |
| Une écriture refusée figure au journal — que faut-il vérifier aussi ? | Le disque. Un contrôle qui journalise mais exécute est pire que rien |
| Un agent se trompe. Pourquoi est-ce pire qu'un assistant qui se trompe ? | L'action a déjà eu lieu |


---
## À vous

1. **Écrivez la politique que votre office utiliserait réellement.** Pas
   `approve_reads_only` — une fonction qui inspecte les arguments. Refusez
   `save_note` hors d'un répertoire nommé ; refusez une recherche dont la requête
   mentionne un identifiant de répondant. Les politiques sont du code, et c'est ce
   qui les rend testables.
2. **Ajoutez un cinquième outil, puis justifiez-le.** Quelque chose dont votre
   office a réellement besoin — une consultation dans une table publiée, par
   exemple. Puis écrivez deux phrases sur ce qu'une demande malveillante ou confuse
   pourrait en faire. Si vous ne pouvez pas écrire ces deux phrases, n'ajoutez pas
   l'outil.
3. **Cassez-le délibérément.** Écrivez un `ScriptedModel` qui demande vingt fois le
   même outil, ou passe des arguments du mauvais type, ou demande un nom de fichier
   `../../etc/passwd`. Corrigez ce qui casse. L'outil `save_note` assainit déjà son
   nom de fichier — lisez cette ligne et décidez si vous lui faites confiance.
4. **Mesurez le coût.** Exécutez l'agent réel sur cinq questions et notez le nombre
   d'appels au modèle pour chacune. C'est cette distribution, et non une moyenne,
   qui sert de base au budget.

Demain matin : l'ingénierie de prompt — rendre un appel unique assez fiable pour
qu'un agent bâti dessus mérite confiance.


---
> ### Si quelque chose n'a pas fonctionné
>
> **Aucun fournisseur de modèle.** Les étapes 1 à 4 et 6 à 8 constituent tout le
> contenu pédagogique et n'en exigent aucun. L'étape 5 démontre que le même code
> accepte un vrai modèle.
>
> **L'agent réel a bouclé sans répondre.** Augmentez `max_steps`, ou resserrez la
> tâche. Un modèle à qui l'on demande trois choses à la fois ne décide souvent
> jamais qu'il a fini.
>
> **L'agent réel a inventé un chiffre.** Lisez ses réponses brutes dans le journal
> d'audit. S'il a énoncé un nombre qu'aucun outil n'a renvoyé, le prompt système
> n'a pas suffi pour ce modèle — c'est un vrai constat, et il a sa place dans votre
> note de vendredi.
>
> **Un appel d'outil a échoué sur un TypeError.** Le modèle a passé un nom
> d'argument inexistant. L'agent lui renvoie l'erreur pour qu'il réessaie ; s'il ne
> s'en remet jamais, vos descriptions d'outils sont ambiguës.
